# Inserting Word Embeddings with a Genetic Algorithm
1.  **Fit the Corpus:** The vector should have a high dot product with its *context* words (words it appears near) and a low dot product with *negative samples* (random words). This is the core idea of Skip-Gram.
2.  **Match the Space:** The vector's **norm** (length) should be similar to the average norm of existing words. This helps it "fit in" to the pre-existing geometric structure.
3.  **Match Semantics:** The vector should be close to known "anchor" words (e.g., we can manually specify that 'beaver' should be near 'animal').

In [2]:
# =============================================================================
# SETUP & IMPORTS
# =============================================================================
import os
import random
import numpy as np
import torch
import sys
import importlib
from pathlib import Path

# add CW2 (parent of embedding_improvements) to path
sys.path.append(str(Path().resolve().parent))

import lab7
import lab6
import lab2

importlib.reload(lab7)
importlib.reload(lab6)
importlib.reload(lab2)

# --- Imports from Previous Labs ---
# We rely on text processing tools from Lab 2 and Lab 6
from lab2 import process_text_network
from lab6 import (
    prepare_visual_genome_text,
    filter_punctuation_from_network,
    analyze_embeddings,
    find_similar_words,
    SkipGramModel,
    ranking_embeddings_signal_to_noise
)

# --- Imports for Lab 7 (Genetic Algorithm) ---
# These functions handle the core Evolutionary Strategy logic,
# data loading for CIFAR-100, and visualization tools.
from lab7 import (
    load_trained_model,
    create_mappings,
    compute_embedding_stats,
    get_cifar100_vocabulary,
    analyze_vocabulary_overlap,
    extract_word_contexts,
    evolve_embedding,           # <--- The core GA function you will implement/use
    visualize_with_inserted_words,
    run_sanity_checks
)

In [3]:
# --- 1. File & Data Paths ---
# Defines where to load the pre-trained model from and where to find
# the corpus text data.
model_path = '../embedding_improvements/best_embedding.pth'
text_file = '../vg_text.txt'
zip_url = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/region_descriptions.json.zip"

In [4]:
# --- 2. Pre-trained Model Parameters ---
# These parameters MUST match the architecture of the 'baseline_model.pth'
# that we are loading.
rare_threshold = 0.00025    # Filtering threshold used to train the model
embedding_dim = 512         # Dimensionality of the embedding vectors
dropout = 0.3              # Dropout rate of the loaded model
punctuation_tokens = {'.', ',', '<RARE>', "'"} # Tokens to ignore

In [5]:
# =============================================================================
# STAGE 1: LOAD CORPUS & BUILD VOCABULARY
# =============================================================================
# In this stage, we load the raw text corpus (Visual Genome) and process
# it to exactly match the vocabulary used to train our baseline model.
#
# We need this vocabulary ('nodes') to:
# 1. Load the pre-trained model (which requires the 'vocab_size').
# 2. Create the 'word_to_idx' and 'idx_to_word' mappings.
# 3. Extract contexts for our new words.

print("\n[STAGE 1] Loading Data & Building Vocabulary")
print("-" * 70)

# --- 1. Download or load text data ---
# We use the Visual Genome dataset as our corpus.
if not os.path.exists(text_file):
    print(f"Text file not found. Downloading corpus from {zip_url}...")
    # This function (from lab6) downloads, extracts, and cleans the text.
    text_file = prepare_visual_genome_text(
        zip_url=zip_url,
        zip_path="region_descriptions.json.zip",
        json_path="region_descriptions.json",
        output_path=text_file
    )
else:
    print(f"✓ Text file '{text_file}' already exists.")

# --- 2. Build co-occurrence network ---
# This step (from lab2) processes the entire text file, counts word
# frequencies, filters rare words, and builds the co-occurrence network.
print("\nProcessing text to build co-occurrence network...")
network_data = process_text_network(
    text_file,
    rare_threshold=rare_threshold,  # Must match the model's training
    rare_token="<RARE>",
    distance_mode="inverted",
    verbose=False  # Set to True to see progress
)

# --- 3. Filter punctuation ---
# We clean up the network by removing punctuation tokens.
network_data = filter_punctuation_from_network(
    network_data,
    punctuation_tokens=punctuation_tokens
)

# --- 4. Finalize Vocabulary ---
# The 'nodes' list from the network IS our final vocabulary.
nodes = network_data['nodes']
vocab_size = len(nodes)

print(f"\n{'-'*70}")
print(f"✓ STAGE 1 Complete: Vocabulary built.")
print(f"  Total vocabulary size (Vocab Size): {vocab_size} nodes")
print(f"  Total graph edges: {network_data['graph'].number_of_edges():,}")


[STAGE 1] Loading Data & Building Vocabulary
----------------------------------------------------------------------
✓ Text file '../vg_text.txt' already exists.

Processing text to build co-occurrence network...

🔧 PUNCTUATION FILTER:
  Removed: {"'", '.', '<RARE>', ','}
  Nodes: 458 → 455
  Edges: 50,128 → 48,764

----------------------------------------------------------------------
✓ STAGE 1 Complete: Vocabulary built.
  Total vocabulary size (Vocab Size): 455 nodes
  Total graph edges: 48,764


In [6]:
# =============================================================================
# STAGE 2: LOAD PRE-TRAINED MODEL & COMPUTE STATS
# =============================================================================
# Now we load the Skip-Gram model that was trained on the vocabulary we
# just built. We also compute the vital statistics of this embedding space.

print("\n[STAGE 2] Loading Model & Analyzing Embedding Space")
print("-" * 70)

# --- 1. Load Model & Embeddings ---
# This function loads the .pth file, initializes the SkipGramModel
# with the *exact* same architecture, and loads the trained weights.
# It then extracts the final (input) embedding matrix as a NumPy array.
model, embeddings = load_trained_model(
    model_path=model_path,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    dropout=dropout
)

# --- 2. Create Mappings ---
# We need fast lookups between words and their corresponding index
# in the embedding matrix.
word_to_idx, idx_to_word = create_mappings(nodes)

# --- 3. Compute Embedding Space Statistics ---
# **Crucial Step for the GA:** We compute the statistics of the *existing*
# embedding space. Our fitness function will use these values to
# ensure that new, evolved vectors "fit in" with the original vectors.
#
# - 'mean_norm': The average length (L2 norm) of all existing vectors.
# - 'std_norm': The standard deviation of the vector lengths.
# - 'global_std': The standard deviation of *all* embedding values.
#                 (Used to scale the GA's mutation strength).
embedding_stats = compute_embedding_stats(embeddings)

print(f"\n{'-'*70}")
print(f"✓ STAGE 2 Complete: Model loaded and stats computed.")
print(f"  Embedding matrix shape: {embeddings.shape}")
print(f"  Key Stats for Fitness Function:")
print(f"    ├─ Mean Vector Norm: {embedding_stats['mean_norm']:.4f}")
print(f"    ├─ Std. Dev of Norms: {embedding_stats['std_norm']:.4f}")
print(f"    └─ Global Std. Dev: {embedding_stats['global_std']:.4f} (for mutation)")


[STAGE 2] Loading Model & Analyzing Embedding Space
----------------------------------------------------------------------
✓ Loaded model: 455 embeddings, dim=512

----------------------------------------------------------------------
✓ STAGE 2 Complete: Model loaded and stats computed.
  Embedding matrix shape: (455, 512)
  Key Stats for Fitness Function:
    ├─ Mean Vector Norm: 1.3635
    ├─ Std. Dev of Norms: 0.1082
    └─ Global Std. Dev: 0.0604 (for mutation)


In [7]:
# =============================================================================
# STAGE 3: SANITY CHECKS (VERIFICATION)
# =============================================================================
# Before we start the complex GA process, let's verify that the model
# we loaded is sane and produces reasonable results.
#
# This function will:
# 1. Check model properties (e.g., it's in eval mode).
# 2. Check embedding matrix stats (e.g., no NaNs/Infs).
# 3. Print nearest neighbors for a few common words ('man', 'dog', etc.)
#    using the `find_similar_words` function from Lab 6.
#
# If the neighbors look sensible, our embedding space is good!

run_sanity_checks(model, embeddings, nodes, word_to_idx)


SANITY CHECKS

1. Model Configuration:
   Training mode: False
   Device: cpu

2. Embedding Quality:
   Shape: (455, 512)
   Mean: 0.000618, Std: 0.060445
   Min: -0.263634, Max: 0.247017
   Contains NaN: False, Contains Inf: False

3. Embedding Norms:
   Mean: 1.3635, Std: 0.1082
   Range: [1.1466, 1.9108]

4. Vocabulary Test:
   'man       ' → idx=   9, norm=1.5022
      Similar: woman(0.894), person(0.888), girl(0.786), child(0.778), lady(0.777)
   'woman     ' → idx=  19, norm=1.5100
      Similar: man(0.894), person(0.862), girl(0.819), lady(0.817), boy(0.806)
   'dog       ' → idx=  64, norm=1.3377
      Similar: horse(0.813), cat(0.811), cow(0.796), bear(0.779), sheep(0.751)
   'car       ' → idx=  45, norm=1.4056
      Similar: vehicle(0.779), van(0.764), bus(0.758), truck(0.749), motorcycle(0.744)
   'blue      ' → idx=  11, norm=1.2745
      Similar: red(0.822), pink(0.768), white(0.767), orange(0.755), purple(0.754)

✓ SANITY CHECKS COMPLETE


In [8]:


print("\n[STAGE 4] Vocabulary Analysis & Target Selection")
print("-" * 70)

# --- 1. Analysis: Compare against CIFAR-100 ---
# Let's see how many CIFAR-100 class names are missing from our
# Visual Genome vocabulary.
cifar_vocab = get_cifar100_vocabulary()

# This function prints a detailed analysis and returns the list of
# words that are in CIFAR-100 but NOT in our 'nodes' (model vocab).
missing_cifar_words = analyze_vocabulary_overlap(cifar_vocab, nodes)



[STAGE 4] Vocabulary Analysis & Target Selection
----------------------------------------------------------------------

Loading CIFAR-100 vocabulary...
✓ CIFAR-100 vocabulary loaded: 100 classes

VOCABULARY OVERLAP ANALYSIS
CIFAR-100 vocabulary: 100 classes
Network vocabulary: 455 words
Overlapping words: 32 (32.0%)
Missing from network: 68

Found: apple, baby, bear, bed, bicycle, bottle, bowl, boy, bridge, bus, can, chair, clock, cloud, couch, cup, elephant, girl, house, keyboard, lamp, man, motorcycle, mountain, mouse, orange, plate, road, table, tank, train, woman

Missing: aquarium_fish, beaver, bee, beetle, butterfly, camel, castle, caterpillar, cattle, chimpanzee, cockroach, crab, crocodile, dinosaur, dolphin, flatfish, forest, fox, hamster, kangaroo, lawn_mower, leopard, lion, lizard, lobster, maple_tree, mushroom, oak_tree, orchid, otter, palm_tree, pear, pickup_truck, pine_tree, plain, poppy, porcupine, possum, rabbit, raccoon, ray, rocket, rose, sea, seal, shark, shrew, sku

/Users/nimunbajwa/Desktop/AI/CW2/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
# This cell defines all the key parameters for our experiment.
#
# -----------------------------------------------------------------------------
# 📖 TUNING NOTE:
# The `ga_...` parameters and `fitness_weights` are the most
# interesting ones to experiment with. Small changes here can
# significantly alter the final position and quality of the
# inserted embeddings.
# -----------------------------------------------------------------------------

anchors = {
    'aquarium_fish': ['water', 'colorful', 'tank', 'animal'],
    'beaver': ['animal', 'fur', 'wood', 'tail'],
    'bee': ['yellow', 'flower', 'flying', 'small'],
    'beetle': ['animal', 'black', 'small', 'ground'],
    'butterfly': ['flower', 'flying', 'colorful', 'small'],
    'camel': ['animal', 'sand', 'standing', 'brown'],
    'castle': ['building', 'tower', 'wall', 'stone'],
    'caterpillar': ['animal', 'leaf', 'green', 'small'],
    'cattle': ['cow', 'animal', 'grass', 'field'],
    'chimpanzee': ['animal', 'fur', 'face', 'hands'],
    'cockroach': ['animal', 'floor', 'dark', 'small'],
    'crab': ['animal', 'water', 'sand', 'legs'],
    'crocodile': ['animal', 'water', 'green', 'tail'],
    'dinosaur': ['animal', 'large', 'tail', 'standing'],
    'dolphin': ['animal', 'water', 'ocean', 'blue'],
    'flatfish': ['animal', 'water', 'ocean', 'grey'],
    'forest': ['trees', 'tree', 'leaves', 'green'],
    'fox': ['animal', 'fur', 'tail', 'dog'],
    'hamster': ['animal', 'small', 'fur', 'mouse'],
    'kangaroo': ['animal', 'tail', 'legs', 'standing'],
    'lawn_mower': ['grass', 'metal', 'wheels', 'ground'],
    'leopard': ['animal', 'fur', 'cat', 'yellow'],
    'lion': ['animal', 'fur', 'cat', 'standing'],
    'lizard': ['animal', 'tail', 'green', 'small'],
    'lobster': ['animal', 'water', 'red', 'legs'],
    'maple_tree': ['tree', 'leaves', 'plant', 'green'],
    'mushroom': ['plant', 'brown', 'ground', 'small'],
    'oak_tree': ['tree', 'leaves', 'wood', 'plant'],
    'orchid': ['flower', 'plant', 'purple', 'pink'],
    'otter': ['animal', 'water', 'fur', 'standing'],
    'palm_tree': ['tree', 'beach', 'sand', 'leaves'],
    'pear': ['fruit', 'green', 'food', 'round'],
    'pickup_truck': ['truck', 'vehicle', 'wheels', 'car'],
    'pine_tree': ['tree', 'green', 'leaves', 'plant'],
    'plain': ['field', 'grass', 'open', 'ground'],
    'poppy': ['flower', 'red', 'plant', 'bright'],
    'porcupine': ['animal', 'fur', 'tail', 'standing'],
    'possum': ['animal', 'fur', 'tail', 'tree'],
    'rabbit': ['animal', 'small', 'fur', 'ears'],
    'raccoon': ['animal', 'fur', 'tail', 'face'],
    'ray': ['animal', 'water', 'large', 'tail'],
    'rocket': ['airplane', 'sky', 'flying', 'air'],
    'rose': ['flower', 'red', 'plant', 'bright'],
    'sea': ['water', 'ocean', 'waves', 'blue'],
    'seal': ['animal', 'water', 'fur', 'sitting'],
    'shark': ['animal', 'water', 'ocean', 'large'],
    'shrew': ['animal', 'small', 'fur', 'ground'],
    'skunk': ['animal', 'fur', 'tail', 'black'],
    'skyscraper': ['building', 'tower', 'city', 'tall'],
    'snail': ['animal', 'small', 'ground', 'round'],
    'snake': ['animal', 'long', 'tail', 'green'],
    'spider': ['animal', 'legs', 'small', 'dark'],
    'squirrel': ['animal', 'tree', 'tail', 'fur'],
    'streetcar': ['train', 'street', 'tracks', 'vehicle'],
    'sunflower': ['flower', 'yellow', 'plant', 'large'],
    'sweet_pepper': ['food', 'green', 'red', 'ground'],
    'telephone': ['phone', 'hand', 'wire', 'table'],
    'television': ['tv', 'screen', 'display', 'monitor'],
    'tiger': ['animal', 'cat', 'fur', 'stripes'],
    'tractor': ['vehicle', 'truck', 'wheels', 'metal'],
    'trout': ['animal', 'water', 'silver', 'small'],
    'tulip': ['flower', 'plant', 'pink', 'red'],
    'turtle': ['animal', 'water', 'green', 'round'],
    'wardrobe': ['cabinet', 'wood', 'room', 'standing'],
    'whale': ['animal', 'water', 'ocean', 'large'],
    'willow_tree': ['tree', 'leaves', 'plant', 'green'],
    'wolf': ['animal', 'dog', 'fur', 'tail'],
    'worm': ['animal', 'small', 'ground', 'long'],
}

if len(anchors) == len(missing_cifar_words):
    print("✓ Using all missing CIFAR-100 words as targets.")
    for a, b in zip(anchors.keys(), missing_cifar_words):
        if a != b:
            print(f"⚠️  Warning: Anchor word '{a}' does not match missing word '{b}'")
else:
    print("⚠️  Warning: Number of anchors does not match number of missing CIFAR-100 words.")

for missing, anchors in anchors.items():
    for a in anchors:
        if a not in nodes:
            print(f"⚠️  Warning: Anchor context word '{a}' for target '{missing}' not in vocabulary.")
#aquarium_fish, beaver, bee, beetle, butterfly, camel, castle, caterpillar, cattle, chimpanzee, cockroach, crab, crocodile, dinosaur, dolphin, flatfish, forest, fox, hamster, kangaroo, lawn_mower, leopard, lion, lizard, lobster, maple_tree, mushroom, oak_tree, orchid, otter, palm_tree, pear, pickup_truck, pine_tree, plain, poppy, porcupine, possum, rabbit, raccoon, ray, rocket, rose, sea, seal, shark, shrew, skunk, skyscraper, snail, snake, spider, squirrel, streetcar, sunflower, sweet_pepper, telephone, television, tiger, tractor, trout, tulip, turtle, wardrobe, whale, willow_tree, wolf, worm


# --- 4. Genetic Algorithm (ES) Parameters ---
context_window = 2         # (Window for context extraction, from Lab 6)

ga_pop_size = 100          # Population size (λ in 1+λ).
                           # We create 100 "offspring" per generation.
ga_generations = 300       # Number of evolutionary cycles to run.
ga_mutation_factor = 0.05  # The "step size" of the mutation. This is
                           # scaled by the embedding space's global
                           # standard deviation.

# --- 5. Fitness Function Weights ---
# **This is the most critical part!**
# These weights define the "goal" of the evolution. They control the
# trade-off between the three parts of our fitness function.
fitness_weights = {
    # 60% priority: Fit the corpus (Skip-Gram objective).
    'corpus': 0.60,

    # 15% priority: Match the "shape" of the space.
    # (i.e., new vectors should have a similar norm/length).
    'norm': 0.15,

    # 25% priority: Be close to our semantic anchors.
    'anchor': 0.25
}

print("✓ All configurations set.")
print(f"  GA Config: {ga_pop_size} population, {ga_generations} generations")
print(f"  Fitness Weights: Corpus={fitness_weights['corpus']}, "
      f"Norm={fitness_weights['norm']}, "
      f"Anchor={fitness_weights['anchor']}")

✓ Using all missing CIFAR-100 words as targets.
✓ All configurations set.
  GA Config: 100 population, 300 generations
  Fitness Weights: Corpus=0.6, Norm=0.15, Anchor=0.25


In [10]:


# This Counter will be used to select the positive samples (ctx_vecs)
# in the fitness function.
target_words = missing_cifar_words

print("\n[STAGE 5] Extracting Corpus Contexts for Target Words")
print("-" * 70)

# We need the 'set(nodes)' to quickly check if a context word is
# actually in our vocabulary. We ignore any context words that aren't.
vocab_set = set(nodes)

contexts = extract_word_contexts(
    text_file=text_file,
    target_words=target_words,
    vocab_set=vocab_set,
    window=context_window
)

print(f"\n{'-'*70}")
print("✓ STAGE 5 Complete: Contexts extracted.")


[STAGE 5] Extracting Corpus Contexts for Target Words
----------------------------------------------------------------------
 Complete 

 Context statistics
  aquarium_fish:      0 contexts,   0 unique words
  beaver    :    147 contexts,  53 unique words
  bee       :    236 contexts,  68 unique words
  beetle    :    210 contexts,  52 unique words
  butterfly :   2472 contexts, 228 unique words
  camel     :    610 contexts, 132 unique words
  castle    :   2543 contexts, 228 unique words
  caterpillar:     91 contexts,  38 unique words
  cattle    :   2555 contexts, 220 unique words
  chimpanzee:     48 contexts,  19 unique words
  cockroach :     34 contexts,  17 unique words
  crab      :    601 contexts, 128 unique words
  crocodile :    154 contexts,  51 unique words
  dinosaur  :    821 contexts, 145 unique words
  dolphin   :    468 contexts, 102 unique words
  flatfish  :      0 contexts,   0 unique words
  forest    :  13727 contexts, 376 unique words
  fox       :    545 c

### Fitness Weights Experiments

In [12]:
from itertools import product
phase_dir = "fitness_func"

fixed_params = {
    "ga_pop_size": 100,
    "ga_generations": 300,
    "ga_mutation_factor": 0.05
}
fitness_weight_lists = [
    {
    "corpus": 0.6,
    "norm": 0.15,
    "anchor": 0.25
    },
    {
    "corpus": 0.5,
    "norm": 0.15,
    "anchor": 0.35
    },
    {
    "corpus": 0.5,
    "norm": 0.25,
    "anchor": 0.25
    },
    {
    "corpus": 0.4,
    "norm": 0.25,
    "anchor": 0.35
    },
    {
    "corpus": 0.3,
    "norm": 0.3,
    "anchor": 0.4
    },
]
for fitness_weights in fitness_weight_lists:   
    corpus = fitness_weights['corpus']
    norm = fitness_weights['norm']
    anchor = fitness_weights['anchor']
    inserted_embeddings_list = []
    ga_config = {
        **fixed_params,
        "fitness_weights": fitness_weights
    }
    # --- Main Evolution Loop ---
    for word in target_words:
        # Evolve the embedding for this single word
        evolved_vec = evolve_embedding(
            word,
            contexts,
            embeddings,       # The original embedding matrix
            word_to_idx,
            nodes,            # The original vocab list (for neg. sampling)
            embedding_stats,  # Stats for the 'norm' fitness term
            anchors,          # Dict of anchors for the 'anchor' fitness term
            ga_config         # All GA hyperparameters
        )
        inserted_embeddings_list.append(evolved_vec)

    inserted_embeddings = np.array(inserted_embeddings_list)
    full_embeddings = np.vstack([embeddings, inserted_embeddings])
    full_embeddings_tensor = torch.tensor(full_embeddings, dtype=torch.float32)
    all_vocab = nodes + target_words
    save_path=f"{phase_dir}/corpus_{corpus}_norm_{norm}_anchor_{anchor}.pth"


    torch.save({
        # The (N+K, D) embedding matrix
        'embeddings': full_embeddings_tensor,

        # The (N+K) vocabulary list
        'vocab': all_vocab,

        # Just the list of words we inserted
        'inserted_words': target_words,

        # Metadata for quick loading
        'metadata': {
            'n_original': len(nodes),
            'n_inserted': len(target_words),
            'vocab_size': len(all_vocab),
            'embedding_dim': inserted_embeddings.shape[1]
        }
    }, save_path)




  Evolving: 'aquarium_fish' G0=0.3310 G50=0.3941 G100=0.4159 G150=0.4415 G200=0.4682 G250=0.4911 ✓ Final=0.5076

  Evolving: 'beaver' G0=0.3580 G50=0.4148 G100=0.4401 G150=0.4650 G200=0.4868 G250=0.5039 ✓ Final=0.5138

  Evolving: 'bee' G0=0.3559 G50=0.4154 G100=0.4398 G150=0.4645 G200=0.4857 G250=0.5034 ✓ Final=0.5142

  Evolving: 'beetle' G0=0.3548 G50=0.4136 G100=0.4389 G150=0.4639 G200=0.4852 G250=0.5014 ✓ Final=0.5126

  Evolving: 'butterfly' G0=0.3632 G50=0.4180 G100=0.4436 G150=0.4686 G200=0.4898 G250=0.5052 ✓ Final=0.5157

  Evolving: 'camel' G0=0.3644 G50=0.4158 G100=0.4404 G150=0.4651 G200=0.4862 G250=0.5029 ✓ Final=0.5138

  Evolving: 'castle' G0=0.3619 G50=0.4163 G100=0.4412 G150=0.4663 G200=0.4874 G250=0.5033 ✓ Final=0.5145

  Evolving: 'caterpillar' G0=0.3549 G50=0.4112 G100=0.4351 G150=0.4595 G200=0.4816 G250=0.4990 ✓ Final=0.5110

  Evolving: 'cattle' G0=0.3622 G50=0.4156 G100=0.4402 G150=0.4654 G200=0.4858 G250=0.5024 ✓ Final=0.5135

  Evolving: 'chimpanzee' G0=0.3582

### Analysis of Fitness Weight Experiments

In [2]:
save_dir = "./fitness_func"
all_signal_to_noise = {}

for filename in sorted(os.listdir(save_dir)):
    if not filename.endswith(".pth"):
        continue
    # --- 1. Load Best Model ---
    # We load the model, nodes, and embeddings saved by train_embeddings
    model_path = os.path.join(save_dir, filename)
    prefix = os.path.splitext(filename)[0]  # Remove .pth extension
    output_txt = os.path.join(save_dir + "/results/", f"{prefix}.txt")

    print(f"\nProcessing {filename}")
    print(f"→ Saving output to {output_txt}")

    original_stdout = sys.stdout
    with open(output_txt, "w") as f:
        sys.stdout = f  # Change the standard output to the file we created.
        if os.path.exists(model_path):
            checkpoint = torch.load(model_path)

            embeddings = checkpoint['embeddings'].cpu().numpy()
            nodes = checkpoint['vocab']
            inserted_words = checkpoint['inserted_words']
            metadata = checkpoint['metadata']
        else:
            print("❌ Error: No saved model found!")
            print("Please run the training cell above before proceeding.")
            # Raise an error to stop execution if the model is missing
            raise FileNotFoundError(f"{model_path} not found. Please train the model first.")

        similarity_examples = target_words
        cluster_seeds = target_words

        analogy_examples = []
        for word in target_words:
            if word in anchors and len(anchors[word]) >= 2:
                # Create an analogy: (anchor1, anchor2, new_word)
                analogy_examples.append((
                    anchors[word][0],  # e.g., 'dog'
                    anchors[word][1],  # e.g., 'animal'
                    word               # e.g., 'wolf'
                ))
            if len(analogy_examples) >= 3:
                break  # Limit to 3 analogies for brevity

        # --- 2. Qualitative Analysis ---
        signal_to_noise_metric =  ranking_embeddings_signal_to_noise(nodes, embeddings)
        print(f"Signal to Noise Metric: {signal_to_noise_metric:.4f}")
        all_signal_to_noise[prefix] = signal_to_noise_metric
        
        analyze_embeddings(
            nodes=nodes, 
            embeddings=embeddings,
            similarity_examples=similarity_examples,
            analogy_examples=analogy_examples,
            cluster_seeds=cluster_seeds
        )

        # --- 3. t-SNE Visualization ---
        # We'll create a 2D plot of the *most frequent* words
        # NOTE: Visualizing all 10k+ words is too slow and unreadable.
        # The function defaults to the top 200, we'll use 300.
        print("\n" + "="*80)
        print("VISUALIZING EMBEDDINGS")
        print("="*80)
        print("Generating t-SNE plot for the top 300 words...")
        print("(This may take a moment...)")

        tsne_file = os.path.join(save_dir + "/results", f"{prefix}_tsne.png")

        visualize_with_inserted_words(
            nodes=nodes,
            embeddings=embeddings,
            inserted_words=target_words,
            output_file=tsne_file,
            sample_size=500  # Sample 500 words, including all our inserted ones
        )
    sys.stdout = original_stdout  # Reset standard output to original value

NameError: name 'os' is not defined

In [1]:
# Top 10 in all_mean_minus_p5, all_cifar_cls_sim -> see if there's anything in common
def top_k(d, k=10):
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:k]

top_signal_to_noise = top_k(all_signal_to_noise, k=20)

print("Top 10 Signal to Noise Metrics:")
for name, score in top_signal_to_noise:
    print(f"{name}: {score:.4f}")    

NameError: name 'all_signal_to_noise' is not defined

### GA experiments

In [ ]:
from itertools import product
phase_dir = "genetic_algo"
inserted_embeddings_list = []

sweep_params = {
    "ga_pop_size": [100, 500, 1000],
    "ga_generations": [300, 700, 1000],
    "ga_mutation_factor": [0.01, 0.05, 0.1, 0.2]
}
fitness_weights = {
    'corpus': 0.6,
    'norm': 0.15,
    'anchor': 0.25
    }

for ps, gen, mf in product(
    sweep_params['ga_pop_size'],
    sweep_params['ga_generations'],
    sweep_params['ga_mutation_factor']):
    ga_config = {
        **fixed_params,
        **fitness_weights
    }

    # --- Main Evolution Loop ---
    for word in target_words:
        # Evolve the embedding for this single word
        evolved_vec = evolve_embedding(
            word,
            contexts,
            embeddings,       # The original embedding matrix
            word_to_idx,
            nodes,            # The original vocab list (for neg. sampling)
            embedding_stats,  # Stats for the 'norm' fitness term
            anchors,          # Dict of anchors for the 'anchor' fitness term
            ga_config         # All GA hyperparameters
        )
        inserted_embeddings_list.append(evolved_vec)

    inserted_embeddings = np.array(inserted_embeddings_list)
    full_embeddings = np.vstack([embeddings, inserted_embeddings])
    full_embeddings_tensor = torch.tensor(full_embeddings, dtype=torch.float32)
    all_vocab = nodes + target_words
    save_path=f"{phase_dir}/corpus_{corpus}_norm_{norm}_anchor_{anchor}.pth",


    torch.save({
        # The (N+K, D) embedding matrix
        'embeddings': full_embeddings_tensor,

        # The (N+K) vocabulary list
        'vocab': all_vocab,

        # Just the list of words we inserted
        'inserted_words': target_words,

        # Metadata for quick loading
        'metadata': {
            'n_original': len(nodes),
            'n_inserted': len(target_words),
            'vocab_size': len(all_vocab),
            'embedding_dim': inserted_embeddings.shape[1]
        }
    }, save_path)